In [40]:
import pandas as pd
import numpy as np
from scipy.stats import qmc
from sklearn.preprocessing import StandardScaler as SS
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
import xgboost as xgb
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import warnings
from sklearn.exceptions import ConvergenceWarning
import itertools

In [ ]:
lista_zavorra = [0, 20, 40, 60]
campioni_per_zavorra = 50

limiti_inferiori = [0.0, -4.5, 0.0, 1.5]
limiti_superiori = [6.0, -2.0, 0.3, 4.5]

risultati = []

#LHS
sampler = qmc.LatinHypercube(d=4, seed=42)

for zavorra in lista_zavorra:
    campioni_lhs = sampler.random(n=campioni_per_zavorra)
    campioni_scalati = qmc.scale(campioni_lhs, limiti_inferiori, limiti_superiori)

    for riga in campioni_scalati:
        vento = round(riga[0], 2)
        coeff_teta = round(riga[1], 2)
        std_vento = round(riga[2], 3)
        delay = round(riga[3], 2)

        teta_rampa = round(vento * coeff_teta, 2)

        risultati.append({
            'vento_medio': vento,
            'std_vento': round(std_vento*vento,2),
            'zavorra': zavorra,
            'delay_secondi': delay,
            'angolo_rampa': teta_rampa,
            'APOGEO_MISURATO': '',
            'ANGOLO_ACCENSIONE_C6': ''
        })

df = pd.DataFrame(risultati)
df.to_csv("Simulazioni_GEMINI.csv", index=False)

In [ ]:
df = pd.read_csv("Simulazioni_GEMINI.csv")

#"DATA CLEANING"
for column in df.columns:
  df[column] = df[column].astype("str")
  df[column] = df[column].str.replace(",", ".")
  df[column] = df[column].str.replace(r'[^0-9.]', '', regex=True)
  df[column] = df[column].apply(lambda x: x.split('.')[0] + '.' + x.split('.')[1] if len(x.split('.')) > 2 else x)
  df[column] = df[column].astype("float")

#SHUFFLE
df = df.sample(frac=1, random_state=999).reset_index(drop=True)

#LABELS
y1 = df["APOGEO_MISURATO"]
y2 = df["ANGOLO_ACCENSIONE_C6"]

#PREDICTIVE VARIABLES
x = df.drop(["APOGEO_MISURATO", "ANGOLO_ACCENSIONE_C6"], axis=1)

#NORMALIZATION
scaler = SS()
x_norm = scaler.fit_transform(x)

In [ ]:
#RANDOM FOREST
modello_rf = RandomForestRegressor(
    n_estimators=100,
    random_state=999,
    n_jobs=-1)

#XGBOOST
modello_xgb = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    random_state=999,
    n_jobs=-1)

#SUPPORT VECTOR REGRESSION
modello_svr = SVR(
    kernel='rbf',
    C=50, #100,
    gamma='scale')

#NN
nn = MLPRegressor(
  hidden_layer_sizes=(16, 8),
  activation='tanh',
  solver='lbfgs',
  alpha=0.1,
  max_iter=5000,
  tol=0.001,
  random_state=999)

modelli_livello_0 = [('m1', modello_rf), ('m2', modello_xgb), ('m3', modello_svr), ('m4', nn)]

In [ ]:
#STACKING
stacking = StackingRegressor(
  estimators=modelli_livello_0,
  final_estimator=Ridge(),
  cv=5,
  n_jobs=-1)

In [42]:
#VALUTAZIONE MODELLO E PREVISIONI
def val(target, reale, modello, dati_x):
    kf = KFold(n_splits=5, shuffle=True, random_state=999)
    previsioni = cross_val_predict(modello, dati_x, reale, cv=kf)
    
    mae = mean_absolute_error(reale, previsioni)
    rmse = np.sqrt(mean_squared_error(reale, previsioni))
    r2 = r2_score(reale, previsioni)
    
    print(f"Risultati modello x previsione {target}")
    print(f"MAE: {mae:.2f}")
    print(f"RMSE: {rmse:.2f}")
    print(f"R^2: {r2:.4f}\n")
    
    return previsioni

In [43]:
print("Cucinando...\n")
prev_apo = val("APOGEO", y1, stacking, x_norm)
prev_ang = val("ANGOLO", y2, stacking, x_norm)

Cucinando...

Risultati modello x previsione APOGEO
MAE: 6.90
RMSE: 10.07
R^2: 0.8716

Risultati modello x previsione ANGOLO
MAE: 8.35
RMSE: 11.67
R^2: 0.2702



In [ ]:
#DATAFRAME ERRORI ASSOLUTI DEL MODELLO
df_boot = pd.DataFrame({
    'std_vento': df['std_vento'],
    'err_apo': np.abs(y1 - prev_apo),
    'err_ang': np.abs(y2 - prev_ang)
})

#DIVISIONE IN FASCE DI TURBOLENZA
bins = [-0.01, 0.4, 0.8, 1.2, 2.2] 
labels = ['Fascia 0.0 - 0.4 m/s', 
    'Fascia 0.4 - 0.8 m/s', 
    'Fascia 0.8 - 1.2 m/s',
    'Fascia 1.2 -2.2 m/s']
df_boot['fascia'] = pd.cut(df_boot['std_vento'], bins=bins, labels=labels)

#BOOTSTRAP
n_bootstraps = 2000
risultati_finali = []

for fascia in labels:
    dati_fascia = df_boot[df_boot['fascia'] == fascia]
        
    boot_percentiles_apo = []
    boot_percentiles_ang = []
    
    for _ in range(n_bootstraps):
        campione = dati_fascia.sample(frac=1.0, replace=True)
        
        #90esimo PERCENTILE DEGLI ERRORI ASSOLUTI
        boot_percentiles_apo.append(np.percentile(campione['err_apo'], 90))
        boot_percentiles_ang.append(np.percentile(campione['err_ang'], 90))
        
    #ESTRAZIONE DEI LIMITI DI CONFIDENZA AL 95%
    apo_lower = np.percentile(boot_percentiles_apo, 2.5)
    apo_upper = np.percentile(boot_percentiles_apo, 97.5)
    apo_mean = np.mean(boot_percentiles_apo)
    
    ang_lower = np.percentile(boot_percentiles_ang, 2.5)
    ang_upper = np.percentile(boot_percentiles_ang, 97.5)
    ang_mean = np.mean(boot_percentiles_ang)
    
    risultati_finali.append({
        'Fascia': fascia,
        'Lanci_nel_dataset': len(dati_fascia),
        'Apo_Medio': apo_mean, 'Apo_Min': apo_lower, 'Apo_Max': apo_upper,
        'Ang_Medio': ang_mean, 'Ang_Min': ang_lower, 'Ang_Max': ang_upper
    })

print("RISULTATI BOOTSTRAP: INTERVALLI DI CONFIDENZA")
for r in risultati_finali:
    print(f"\n {r['Fascia']} (Basato su {r['Lanci_nel_dataset']} lanci originali):")
    
    print(f"Errore Max Apogeo atteso: {r['Apo_Medio']:.1f} m")
    print(f"Il 90esimo percentile sta al 95% tra ±{r['Apo_Min']:.1f}m e ±{r['Apo_Max']:.1f}m")

    print(f"Errore Max Angolo atteso: {r['Ang_Medio']:.1f}°")
    print(f"Il 90esimo percentile sta al 95% tra ±{r['Ang_Min']:.1f}° e ±{r['Ang_Max']:.1f}°")

RISULTATI BOOTSTRAP: INTERVALLI DI CONFIDENZA

 Fascia 0.0 - 0.4 m/s (Basato su 109 lanci originali):
Errore Max Apogeo atteso: 10.6 m
Il 90esimo percentile sta al 95% tra ±7.2m e ±16.2m
Errore Max Angolo atteso: 12.8°
Il 90esimo percentile sta al 95% tra ±9.3° e ±17.3°

 Fascia 0.4 - 0.8 m/s (Basato su 32 lanci originali):
Errore Max Apogeo atteso: 23.3 m
Il 90esimo percentile sta al 95% tra ±14.6m e ±31.7m
Errore Max Angolo atteso: 25.8°
Il 90esimo percentile sta al 95% tra ±17.7° e ±32.6°

 Fascia 0.8 - 1.2 m/s (Basato su 28 lanci originali):
Errore Max Apogeo atteso: 14.1 m
Il 90esimo percentile sta al 95% tra ±11.6m e ±19.9m
Errore Max Angolo atteso: 16.5°
Il 90esimo percentile sta al 95% tra ±13.0° e ±22.5°

 Fascia 1.2 -2.2 m/s (Basato su 31 lanci originali):
Errore Max Apogeo atteso: 22.5 m
Il 90esimo percentile sta al 95% tra ±16.5m e ±28.8m
Errore Max Angolo atteso: 24.4°
Il 90esimo percentile sta al 95% tra ±20.1° e ±30.0°


In [ ]:
warnings.filterwarnings("ignore", category=ConvergenceWarning)

#ADDENSTRAMENTO FINALE
modello_apogeo = StackingRegressor(estimators=modelli_livello_0, final_estimator=Ridge(), cv=5)
modello_angolo = StackingRegressor(estimators=modelli_livello_0, final_estimator=Ridge(), cv=5)
modello_apogeo.fit(x_norm, y1)
modello_angolo.fit(x_norm, y2)

#SALVATAGGIO MODELLI
joblib.dump(modello_apogeo, "modello_apogeo.pkl")
joblib.dump(modello_angolo, "modello_angolo.pkl")
joblib.dump(scaler, 'scaler.pkl')

['scaler.pkl']

In [ ]:
#CARICAMENTO MODELLI --> RIPARTIRE DA QUI SE I MODELLI SONO GIA' STATI ADDENSTRATI
modello_apo = joblib.load('modello_apogeo.pkl')
modello_ang = joblib.load('modello_angolo.pkl')
scaler_fisso = joblib.load('scaler.pkl')